## Notebook10

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
us_pop = (
    pl.read_csv(ub + "data/us_city_population.csv")
    .select(c.city, c.state, c.year, c.population)
    .filter(c.population.null_count().over(c.city) <= 1)
)

### Questions

This week's data is the population of 289 American cities at every census from
1790 to 2010, which makes 6,647 rows. Each row is one city in one census year.
The `population` column is in thousands, so New York City in 2010 is 8175.133 and
not 8,175,133. A city that did not exist yet has a population of 0 rather than a
missing value, which matters more than it sounds like it should.

The cities are roughly the largest ones in the country today, traced backward
through time. That means the table is not a sample of American cities and it is
certainly not all of them: 46 states appear, some of them with a single city.
Keep that in mind whenever you compute a total, because the total is over the
cities that happen to be here.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Start with the tools you already have. For the 2010 census only, compute each
state's total city population and the number of cities it contributes, and sort
the states from largest to smallest.

California leads with 71 cities and 18.2 million people in them. Now suppose you
want each city's share of its own state's total. The share needs two numbers on
the same row: the city's population, which you have, and the state total, which is
sitting in a different table with one row per state. Aggregation put it there and
aggregation cannot get it back.

2. `.over()` can. Take the 2010 census, write each state's total onto every city in
that state, and compute the share. Sort by share, largest first.

The top of the result is a column of 100.0. Anchorage is 100 percent of Alaska,
Little Rock is 100 percent of Arkansas, and so on down a list of states that
contribute exactly one city to the table. The share is a share of the cities in
*this table*, and question 1's `n` column showed that for those states the table
holds one city.

3. So look at the states that have enough cities for the share to mean anything.
Keep only the states with at least five cities in the table, then compute the same
share. The count is itself a window: `pl.len().over(c.state)` gives every row the
number of rows in its state, and a window expression can go straight inside a
`.filter()`.

New York City is 90 percent of New York's urban population and Chicago is 72
percent of Illinois's. Nothing is added to the table in the filter and nothing is
removed from it in the `with_columns`. The same expression does different work
depending on which context you drop it into.

4. Here is a comparison that needs the rows preserved. Keep the 2010 cities whose
population is above the mean city population of their own state.

Seventy-six cities of 289, which is barely a quarter of them. That is what a mean
does when one city in the state is enormous: Los Angeles drags California's average
up past most of the state, and the average stops describing a typical city. Swap
`.mean()` for `.median()` and 132 cities come back instead, much nearer to half, as
you would expect from a median.

5. Every window so far has grouped by state and stayed inside one year. Turn it
around: group by *year* and look across the states. Give each city its share of all
289 cities' population in its census year, then follow New York City down the
column.

New York in 2010 is nearly twice the size it was in 1910, yet its share peaked at
19.2 percent of these cities in 1910 and has fallen to 9.6 percent since. Both
statements come from the same two columns. The city got bigger; the country's other
cities got bigger faster.

6. Now for the ordered windows. Sort the table by city and year and compute how
much each city gained since the previous census by subtracting the shifted column,
with `.over(c.city)` so that the shift restarts at each city. Then drop the nulls,
sort by the change, and see which decade of which city added the most people.

The first census of every city now has a `null` change, because there is no row
above it to subtract. That is why the `.drop_nulls()` is there: without it Polars
sorts those nulls to the top of a descending sort and you read a screen of blanks.
What is left is New York City adding 1.9 million people
between 1890 and 1900, then 1.3 million more in the next ten years, with Chicago,
Los Angeles, and Detroit arriving in the 1920s. Nine of the ten largest gains happen
between 1880 and 1930, when the industrial cities were growing. The tenth is New
York again, in the 1990s.

7. Sort the other way and you get the second half of that story. Keep only the 2010
rows with a negative change, so the question becomes: which cities lost people
between 2000 and 2010?

Sixty-one cities shrank in that decade. Detroit lost 237,000 people and Chicago
200,000, part of a long decline. New Orleans lost 141,000 to a single hurricane in
August 2005, which falls between two census counts taken five years on either side
of it. A decade is a coarse instrument, and the column does not tell you which of
these two kinds of loss you are looking at.

8. Absolute change favors big cities, so try growth as a percentage instead. In
the first block, use the same shifted column as a denominator, drop the nulls, and
sort with the fastest growth first. Look at what comes back. In the second block,
save the shifted column as `prev` first, keep only the rows where `prev > 0`, and
then compute the growth and sort again.

**Answer**:


The second block drops only the rows with a zero denominator, and every remaining
row still compares itself to the census immediately before it. At the top is Warren,
Michigan: 700 people in 1950, 89,246 in 1960, a 12,649 percent decade. That one is
real, and it is what a Detroit suburb looked like when Detroit was still growing.
Percent growth rewards a small denominator, so it hands you villages that became
suburbs and it will never once mention New York, while question 6 hands you nothing
but New York. Both are correct. Choose the one that matches the question you were
asked. Notice too that the filter comes after the window: filtering out the zeros
first would change what "the row above" means, and the shift would silently compare
censuses decades apart.

9. One city in this table has an actual hole in it. Portland, Oregon has no
population recorded for 1850. Sort by city and year, then use `.forward_fill()` and
`.backward_fill()`, both scoped with `.over(c.city)`, to fill the gap from above and
from below, and look at Portland from 1830 to 1870.

Forward fill copies 1840 down and says Portland had nobody in it. Backward fill
copies 1860 up and says it had 2,874 people. The census counted 821. Both fills are
guesses, wrong in opposite directions, with the true value between them. That is
generally the most you can expect from a fill. Note also that the `.over()`
changed nothing here, since the row above Portland's gap happens to be Portland. It
would have changed everything if the gap had landed on a city's first row, and you do
not get to know that in advance.